In this file I want to test a smaller grid of porosity ($f$) values. I also want this notebook to be able to work for different custom values, rather than my other file is very pre-set 

### Load intro stuff

In [1]:
%run "/Users/audreyburggraf/Desktop/QUEEN'S/THESIS RESEARCH/PLOTTING C29 989/constants.py"

/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (
/opt/anaconda3/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3432: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/anaconda3/lib/python3.9/site-packages/numpy/core/_methods.py:190: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)
/opt/anaconda3/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3432: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/anaconda3/lib/python3.9/site-packa

In [2]:
%run "/Users/audreyburggraf/Desktop/QUEEN'S/THESIS RESEARCH/PLOTTING C29 989/FUNCTIONS/load_functions"

fortran mie routines unavailable


/opt/anaconda3/lib/python3.9/site-packages/dsharp_opac/dsharp_opac.py:47: UserWarning: could not import compiled mie code - mie calculation will be slow
  warnings.warn(


In [3]:
all_bands    = ["Band 4 nterms2",  "Band 5", "Band 5 robust -1",     "Band 6", "Band 7 nterms2"]
bands_naming = ['Band4_nterms2',   'Band5',  "Band5_robust_minus1",  'Band6',  'Band7_nterms2']
bands_labels = ['Band 4',          'Band 5', "Band 5 robust -1",      'Band 6', 'Band 7']


lambda_bands_cm = mm_to_cm([lambda_mm[b] for b in all_bands])

In [4]:
df_POLF = load_POLF(all_bands, print_things = True)

df_POLF

Band = Band 4 nterms2
Band = Band 5
Band = Band 5 robust -1
Band = Band 6
Band = Band 7 nterms2
 in load_POLF, results = {'Band': ['Band 4 nterms2', 'Band 5', 'Band 5 robust -1', 'Band 6', 'Band 7 nterms2'], 'POLF_Gaussian': [1.1877280000000001, 0.9329613999999999, 1.3769860999999999, 1.4453122, 0.8302132], 'POLF_maxPOLI': [1.561384, 1.0520775, 1.5137895000000001, 1.4862873, 1.0841521], 'POLF_maxStokesI': [1.3677061000000001, 0.9908963, 1.4034492, 1.4639224, 1.0421403]}


,Band,POLF_Gaussian,POLF_maxPOLI,POLF_maxStokesI,POLF_mean
0,Band 4 nterms2,1.187728,1.561384,1.367706,1.372273
1,Band 5,0.932961,1.052077,0.990896,0.991978
2,Band 5 robust -1,1.376986,1.513790,1.403449,1.431408
3,Band 6,1.445312,1.486287,1.463922,1.465174
4,Band 7 nterms2,0.830213,1.084152,1.042140,0.985502


In [10]:
POLF_type = 'max Stokes I'
plot_sf = 1.5

POLF_obs_for_plot = dict(zip(df_POLF["Band"], df_POLF["POLF_maxStokesI"]))
POLF_obs_for_plot

{'Band 4 nterms2': 1.3677061000000001,
 'Band 5': 0.9908963,
 'Band 5 robust -1': 1.4034492,
 'Band 6': 1.4639224,
 'Band 7 nterms2': 1.0421403}

In [5]:
# Set up wavelength array
logwave_vals = np.linspace(0.1, 4, 10000)

lambda_dist_micron = 10**logwave_vals

lambda_dist_cm = micron_to_cm(lambda_dist_micron)

In [6]:
# a_max values for this plot
a_max_test_micron = [1, 100]
a_max_test_cm = micron_to_cm(a_max_test_micron)

In [7]:
def RUN(f, af_min_bound_micron, af_max_bound_micron, N_grains_f):
    a_max_f_dist_cm = micron_to_cm((1/f) * np.linspace(af_min_bound_micron/f,
                        af_max_bound_micron/f,
                        N_grains_f))


    # Run DSHARP for this f
    P, omega, P_times_omega = run_DSHARP(f, a_max_test_cm, a_max_f_dist_cm, lambda_bands_cm, lambda_dist_cm)

    # Prepare data dictionary
    data = {
        "a_max_f_micron": a_max_f_dist_cm * 1e4
    }

    # Add per-band columns
    for i, b in enumerate(bands_naming):  # bands = [4, 5, 6, ...]
        data[f"P_{b}"] = P[:, i]
        data[f"omega_{b}"] = omega[:, i]
        data[f"P_times_omega_{b}"] = P_times_omega[:, i]

    # Convert to DataFrame
    df = pd.DataFrame(data)
    
    return df

### Make data

In [8]:
f = 0.9

af_min_bound_micron = 50
af_max_bound_micron = 501
N_grains_f = (501 - 50) + 1


df_09 = RUN(f, af_min_bound_micron, af_max_bound_micron, N_grains_f)

f_str = str(f).replace('.', '_')  # e.g., 0.3 → '0_3' for filename
file_name = f"P_omega_vs_amax_f_{f_str}_CustomBounds.csv"
df_09.to_csv(P_omega_data_folder_path + file_name, index=False)

print(f"Saved {file_name}")

Please cite Warren & Brandt (2008) when using these optical constants
Please cite Draine 2003 when using these optical constants
Reading opacities from troilitek
Please cite Henning & Stognienko (1996) when using these optical constants
Reading opacities from organicsk
Please cite Henning & Stognienko (1996) when using these optical constants
| material                            | volume fractions | mass fractions |
|-------------------------------------|------------------|----------------|
| Water Ice (Warren & Brandt 2008)    | 0.3642           | 0.2            |
| Astronomical Silicates (Draine 2003)| 0.167            | 0.329          |
| Troilite (Henning)                  | 0.02578          | 0.07434        |
| Organics (Henning)                  | 0.443            | 0.3966         |
using Maxwell-Garnett mixing: first component should be host material (= matrix)
    matrix = Vacuum
Mie ... Done!
Mie ... Done!


In [19]:
f = 0.89

af_min_bound_micron = 50
af_max_bound_micron = 501
N_grains_f = (501 - 50) + 1


df = RUN(f, af_min_bound_micron, af_max_bound_micron, N_grains_f)

f_str = str(f).replace('.', '_')  # e.g., 0.3 → '0_3' for filename
file_name = f"P_omega_vs_amax_f_{f_str}_CustomBounds.csv"
df.to_csv(P_omega_data_folder_path + file_name, index=False)

print(f"Saved {file_name}")

Please cite Warren & Brandt (2008) when using these optical constants
Please cite Draine 2003 when using these optical constants
Reading opacities from troilitek
Please cite Henning & Stognienko (1996) when using these optical constants
Reading opacities from organicsk
Please cite Henning & Stognienko (1996) when using these optical constants
| material                            | volume fractions | mass fractions |
|-------------------------------------|------------------|----------------|
| Water Ice (Warren & Brandt 2008)    | 0.3642           | 0.2            |
| Astronomical Silicates (Draine 2003)| 0.167            | 0.329          |
| Troilite (Henning)                  | 0.02578          | 0.07434        |
| Organics (Henning)                  | 0.443            | 0.3966         |
using Maxwell-Garnett mixing: first component should be host material (= matrix)
    matrix = Vacuum
Mie ... Done!
Mie ... Done!
Saved P_omega_vs_amax_f_0_89_CustomBounds.csv


In [20]:
f = 0.85

af_min_bound_micron = 50
af_max_bound_micron = 501
N_grains_f = (501 - 50) + 1


df = RUN(f, af_min_bound_micron, af_max_bound_micron, N_grains_f)

f_str = str(f).replace('.', '_')  # e.g., 0.3 → '0_3' for filename
file_name = f"P_omega_vs_amax_f_{f_str}_CustomBounds.csv"
df.to_csv(P_omega_data_folder_path + file_name, index=False)

print(f"Saved {file_name}")

Please cite Warren & Brandt (2008) when using these optical constants
Please cite Draine 2003 when using these optical constants
Reading opacities from troilitek
Please cite Henning & Stognienko (1996) when using these optical constants
Reading opacities from organicsk
Please cite Henning & Stognienko (1996) when using these optical constants
| material                            | volume fractions | mass fractions |
|-------------------------------------|------------------|----------------|
| Water Ice (Warren & Brandt 2008)    | 0.3642           | 0.2            |
| Astronomical Silicates (Draine 2003)| 0.167            | 0.329          |
| Troilite (Henning)                  | 0.02578          | 0.07434        |
| Organics (Henning)                  | 0.443            | 0.3966         |
using Maxwell-Garnett mixing: first component should be host material (= matrix)
    matrix = Vacuum
Mie ... Done!
Mie ... Done!
Saved P_omega_vs_amax_f_0_85_CustomBounds.csv


In [24]:
csv_files = {
    0.9:   "P_omega_vs_amax_f_0_9_CustomBounds.csv",
    0.89:   "P_omega_vs_amax_f_0_89_CustomBounds.csv",
    0.85:   "P_omega_vs_amax_f_0_85_CustomBounds.csv",
    0.5:   "P_omega_vs_amax_f_0_5_CustomBounds.csv",
    0.25:   "P_omega_vs_amax_f_0_25_CustomBounds.csv",
    0.3: "P_omega_vs_amax_f_0_3_CustomBounds.csv",
    0.1: "P_omega_vs_amax_f_0_1_CustomBounds.csv",
    0.01:"P_omega_vs_amax_f_0_01_CustomBounds.csv"
}

## Run Fits

In [25]:
bands_fit = ["Band 4 nterms2",  "Band 5 robust -1", "Band 6", "Band 7 nterms2"]

In [28]:
a_max_f_dist_micron_all, P_times_omega_all, best_a_max_f_all, POLF_obs_all, best_sf_all, best_idx_all = find_sf_v3(
    all_bands,
    bands_naming,
    bands_fit,
    csv_files,
    P_omega_data_folder_path,
    df_POLF,
    POLF_type,
    print_results = False
)


# import pandas as pd

# Make dataframe
df_results = pd.DataFrame({
    'f': list(best_a_max_f_all.keys()),
    'best_a_max_micron': list(best_a_max_f_all.values()),
    'best_sf': list(best_sf_all.values())
})

# Optional: sort by f
df_results = df_results.sort_values('f', ascending=False)

# Display
df_results

The Bands we are looking at are: ['Band 4 nterms2', 'Band 5', 'Band 5 robust -1', 'Band 6', 'Band 7 nterms2']
The Bands included in the fit are: ['Band 4 nterms2', 'Band 5 robust -1', 'Band 6', 'Band 7 nterms2']


,f,best_a_max_micron,best_sf
0,0.90,181.481481,1.637554
1,0.89,184.320162,1.648104
2,0.85,191.003460,1.629172
3,0.50,267.735471,1.535748
5,0.30,375.640169,1.516089
4,0.25,430.941884,1.511522
6,0.10,1793.587174,1.626993
7,0.01,278557.114228,1.481296


## Plot data